# Lab 3 Track B — End-to-end offer generation demo

This notebook builds a FAISS index from the local NorthStar policy corpus, retrieves policy context for one supplied customer case, and calls Bedrock to generate a retention offer. Run it from the project environment with AWS credentials configured. The saved cell outputs are the evidence; use **Restart Kernel and Run All** once before committing.

The RAGAS scores are diagnostic under the Fall 2026 update. The repeated four-case comparison and metric analysis are saved separately in `docs/lab3-evidence/rag_eval.csv` and `docs/lab3-ragas-summary.csv`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'rag').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'rag').is_dir():
    raise FileNotFoundError('Open/run this notebook from the cs401r-mking38 repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Repository: {ROOT}')
print(f'Policy corpus exists: {(ROOT / "northstar-policy-docs").is_dir()}')

Repository: /Users/mattking/dev_ta/cs401r-mking38
Policy corpus exists: True


In [2]:
from evaluation.evaluation_harness import OFFER_TEST_CASES
from rag.index import build_index
from rag.offer import GEN_MODEL, generate_offer

index, chunks = build_index(str(ROOT / 'northstar-policy-docs'))
print(f'Indexed {len(chunks)} policy chunks; FAISS vectors: {index.ntotal}')

/Users/mattking/dev_ta/cs401r-mking38/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed 55 policy chunks; FAISS vectors: 55


In [3]:
# Use a supplied, reproducible test case rather than inventing a profile.
case = OFFER_TEST_CASES[0]
result = generate_offer(
    index, chunks, case.customer_context, case.question,
    strategy='two_pass', signal_mode='interpreted',
)

print(f'Customer: {case.customer_id} | Tier: {case.customer_context["loyalty_tier"]}')
print(f'Request: {case.question}')
print('\nGenerated offer:\n')
print(result['answer'])

Customer: CUST-10000317 | Tier: Gold
Request: Generate a personalized retention offer for this at-risk Gold member who shops mostly online for outdoor gear and footwear.

Generated offer:

Subject: Your Gold Member Perks Are Waiting—Free Returns on Your Next Order

Body: As a Gold member, you enjoy free return shipping on every order—a benefit that makes trying new outdoor gear and footwear risk-free online. We'd love to see you back soon and help you find your next adventure. Shop now and return anything within 30 days at no cost.


In [4]:
# Show the retrieved policy sources so the offer can be reviewed for grounding.
for number, context in enumerate(result['contexts'], start=1):
    print(f'--- Retrieved context {number} ---')
    print(context)
    print()

--- Retrieved context 1 ---
[POL-LOY-011 | 4. Tier Benefits]
## 4. Tier Benefits

| Benefit | Bronze | Silver | Gold | Platinum |
|---|---|---|---|---|
| Points earn rate | 1× | 1.25× | 1.5× | 2× |
| Birthday reward | $5 | $10 | $20 | $40 |
| Free standard shipping | $75+ | $50+ | Always | Always |
| Free return shipping | No | No | Yes | Yes |
| Return window | 30 days | 30 days | 30 days | **60 days** |
| Early access to sales | No | 12 hours | 24 hours | 48 hours |
| Free gear repair | No | No | 2 per year | Unlimited |
| Dedicated support line | No | No | No | Yes |
| Annual gear credit | No | No | No | $100 |

Benefits are tied to the member's tier at the time the benefit is used, except the return window, which is fixed at the tier held **at the time of purchase** (see POL-RET-004 §1.1).

--- Retrieved context 2 ---
[KB-FAQ-022 | Loyalty]
## Loyalty

**How do I move up a tier?**
Tier is based on trailing 12-month merchandise spend: Bronze under $500, Silver $500, Gold $2,000, Pla

In [5]:
print(f'Generation model: {GEN_MODEL}')
print('Bedrock token usage:', result['usage'])
print('Retrieved context count:', len(result['contexts']))

Generation model: us.anthropic.claude-haiku-4-5-20251001-v1:0
Bedrock token usage: {'inputTokens': 2852, 'outputTokens': 85, 'totalTokens': 2937, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}
Retrieved context count: 8


In [6]:
# Display the repeated before/after rubric evidence when it is available.
import pandas as pd

summary_path = ROOT / 'docs' / 'lab3-ragas-summary.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print(f'Run `python rag/evaluate.py` to create {summary_path.relative_to(ROOT)}.')

,condition,customer_id,metric,mean,min,max,range,valid_n,expected_n,judge_model
0,after_profile_context,CUST-10000122,faithfulness,0.557692,0.500000,0.615385,0.115385,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
1,after_profile_context,CUST-10000122,answer_relevancy,0.266140,0.258907,0.273372,0.014465,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
2,after_profile_context,CUST-10000122,context_recall,0.333333,0.333333,0.333333,0.000000,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
3,after_profile_context,CUST-10000317,faithfulness,0.323864,0.272727,0.375000,0.102273,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
4,after_profile_context,CUST-10000317,answer_relevancy,0.429443,0.312366,0.546521,0.234155,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
5,after_profile_context,CUST-10000317,context_recall,1.000000,1.000000,1.000000,0.000000,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
6,after_profile_context,CUST-10000588,faithfulness,0.333333,0.222222,0.444444,0.222222,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
7,after_profile_context,CUST-10000588,answer_relevancy,0.297195,0.295435,0.298954,0.003519,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
8,after_profile_context,CUST-10000588,context_recall,0.625000,0.500000,0.750000,0.250000,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
9,after_profile_context,CUST-10000904,faithfulness,0.439394,0.333333,0.545455,0.212121,2,2,us.anthropic.claude-haiku-4-5-20251001-v1:0
